In [2]:
import numpy as np
from sklearn.preprocessing import normalize
from scipy.sparse import csr_matrix, coo_matrix, hstack
import pandas as pd

In [3]:
import pyarrow.parquet as pq
interactions_path = '/Users/alesy/kursach/data/interactions_audio_v2.parquet'
pf = pq.ParquetFile(interactions_path)
raw = next(pf.iter_batches(batch_size=600_000)).to_pandas()
raw.head(2)

,user_id,song_id,play_count,deezer_id,name,artists,genre_root,genre_sub,duration,release_date,...,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,timeSignature,valence
0,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOAKIMP12A8C130995,1,3834123571,The Cove,Jack Johnson,NaN,NaN,112,NaN,...,0.23,0.57,7,0.12,-15.35,1,0.05,123.61,4,0.38
1,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOBBMDR12A8C13253B,2,90270219,Entre Dos Aguas - Remastered 2014,Paco de Lucía,"latin, traditional","latin, worldwide",362,2014-01-01T00:00:00+00:00,...,0.92,0.83,4,0.08,-3.91,0,0.03,102.59,4,0.92


In [4]:
NUM_COLS = ['acousticness', 'danceability', 'energy', 'instrumentalness', 'key','liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'timeSignature',
'valence']
CAT_COLS = ['genre_root','key','mode','timeSignature']

In [5]:
genre_oh = raw['genre_root'].str.get_dummies(sep=', ')
key_oh = pd.get_dummies(raw['key'].astype(int), prefix='key')  # 12 колонок: key_0 ... 

In [6]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
raw[NUM_COLS] = scaler.fit_transform(raw[NUM_COLS])

## final df

In [7]:
df = pd.concat([raw[['user_id', 'song_id', 'play_count']], genre_oh, key_oh, raw[NUM_COLS]], axis=1)
df.head()

,user_id,song_id,play_count,alternative,blues,classical,country,disco,edm,electro,...,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,timeSignature,valence
0,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOAKIMP12A8C130995,1,0,0,0,0,0,0,0,...,-2.058018,1.835941,0.497780,-0.479151,-2.108229,0.706683,-0.313127,0.037398,0.176135,-0.494321
1,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOBBMDR12A8C13253B,2,0,0,0,0,0,0,0,...,1.093787,2.868997,-0.340834,-0.709361,0.977903,-1.415063,-0.543766,-0.713622,0.176135,1.693036
2,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOBXHDL12A81C204C0,1,0,0,0,0,0,0,0,...,0.088864,-0.428838,-1.179448,0.556797,-0.524698,0.706683,0.494111,-0.664317,0.176135,-0.089255
3,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOBYHAJ12A6701BF1D,1,1,0,0,0,0,0,0,...,-1.875304,-0.428838,0.777318,0.787008,-1.541719,0.706683,-0.543766,-0.190910,0.176135,-0.615841
4,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SODACBL12A8C13C273,1,1,0,0,0,0,0,0,...,1.093787,-0.428838,-0.340834,0.326587,0.945531,0.706683,-0.428446,0.480078,0.176135,0.153785


## разделяем interactions на test и train

In [8]:
# --- train/test split: 20% взаимодействий каждого юзера в test ---
import numpy as np

df_shuf = df.sample(frac=1.0, random_state=42).reset_index(drop=True)
df_shuf['rank'] = df_shuf.groupby('user_id').cumcount()
df_shuf['cnt']  = df_shuf.groupby('user_id')['rank'].transform('max') + 1
is_test = df_shuf['rank'] < (df_shuf['cnt'] * 0.2).astype(int)

train = df_shuf[~is_test].reset_index(drop=True)
test  = df_shuf[is_test].reset_index(drop=True)
print(f'train={len(train):,}  test={len(test):,}  users_in_test={test.user_id.nunique():,}')


train=489,694  test=110,306  users_in_test=22,328


In [9]:
# матрица train × features, взвешенная на play_count
FEATURES = [c for c in train.columns if c not in ('user_id', 'song_id', 'play_count', 'rank', 'cnt')]
X = train[FEATURES].values                      # (n_rows, n_features)
w = train['play_count'].values.reshape(-1, 1)   # (n_rows, 1)
Xw = X * w                                      # каждую строку умножили на её play_count

# суммы по юзеру и суммы весов по юзеру
user_ids = train['user_id'].values
users_unique, inv = np.unique(user_ids, return_inverse=True)

num = np.zeros((len(users_unique), X.shape[1]), dtype=np.float64)
den = np.zeros(len(users_unique), dtype=np.float64)
np.add.at(num, inv, Xw)
np.add.at(den, inv, w.ravel())

user_matrix = num / den[:, None]                # (n_users, n_features)
user_emb = pd.DataFrame(user_matrix, index=users_unique, columns=FEATURES)
user_emb.head()


,alternative,blues,classical,country,disco,edm,electro,folk,hip hop,holiday,...,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,timeSignature,valence
00003a4459f33b92906be11abe0e93efc423c0ff,0.214286,0.0,0.0,0.0,0.0,0.0,0.071429,0.0,0.071429,0.0,...,-0.194994,-0.207468,-0.640339,0.433470,-0.216008,0.555129,-0.313127,-0.415287,-0.051563,0.422865
00005c6177188f12fb5e2e82cdbd93e8a3f35e64,0.500000,0.0,0.0,0.0,0.0,0.0,0.125000,0.0,0.000000,0.0,...,0.123123,-0.408971,-0.725198,0.211481,-0.155117,-0.354190,-0.053658,0.434926,0.176135,-0.793057
0007c0e74728ca9ef0fe4eb7f75732e8026a278b,0.714286,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,0.023609,-0.428838,-0.300900,-0.577812,0.253387,0.706683,-0.379024,-0.393135,0.176135,-0.152908
000b474f815bcff17a4bc9ce5324f9352dafe07d,0.750000,0.0,0.0,0.0,0.0,0.0,0.166667,0.0,0.000000,0.0,...,1.135659,0.137357,-0.736846,0.034027,0.780299,0.353058,-0.111317,1.095537,0.176135,0.336065
000b4e4134d5f77d7608fbf86fb3e1adac4478a8,0.818182,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,0.192678,-0.403553,-0.061296,-0.641344,-0.180132,-1.029291,-0.522799,-0.063650,0.176135,0.356318


In [10]:
print(user_emb.shape)                # (n_users, n_features)
print(user_emb.index[:5])            # строковые user_id
print(user_emb.describe().loc[['mean','std']].T.head())


(24064, 49)
Index(['00003a4459f33b92906be11abe0e93efc423c0ff',
       '00005c6177188f12fb5e2e82cdbd93e8a3f35e64',
       '0007c0e74728ca9ef0fe4eb7f75732e8026a278b',
       '000b474f815bcff17a4bc9ce5324f9352dafe07d',
       '000b4e4134d5f77d7608fbf86fb3e1adac4478a8'],
      dtype='str')
                 mean       std
alternative  0.372532  0.264695
blues        0.007993  0.040876
classical    0.023695  0.072111
country      0.043077  0.112243
disco        0.007081  0.039726


## distance methods

In [11]:
def cosine_similarity(A, B):
    sc_prod = A @ B.T
    A_norm = np.linalg.norm(A, axis=1) + 1e-9
    B_norm = np.linalg.norm(B, axis=1) + 1e-9
    return sc_prod / A_norm[:, None] / B_norm[None, :]

def euclideam_dist(A, B):
    # ||a-b||^2 = ||a||^2 + ||b||^2 - 2 a·b — без хранения тензора (n_a, n_b, d)
    A_sq = (A ** 2).sum(axis=1)
    B_sq = (B ** 2).sum(axis=1)
    cross = A @ B.T
    return np.sqrt(np.maximum(A_sq[:, None] + B_sq[None, :] - 2 * cross, 0.0))

def manhattan_dist(A, B, chunk=256):
    # чанкуем по строкам A: иначе intermediate (n_a, n_b, d) не помещается в память
    A32, B32 = A.astype(np.float32), B.astype(np.float32)
    out = np.empty((A.shape[0], B.shape[0]), dtype=np.float32)
    for i in range(0, A.shape[0], chunk):
        out[i:i+chunk] = np.abs(A32[i:i+chunk, None, :] - B32[None, :, :]).sum(axis=-1)
    return out

def chebyshev(A, B, chunk=256):
    A32, B32 = A.astype(np.float32), B.astype(np.float32)
    out = np.empty((A.shape[0], B.shape[0]), dtype=np.float32)
    for i in range(0, A.shape[0], chunk):
        out[i:i+chunk] = np.abs(A32[i:i+chunk, None, :] - B32[None, :, :]).max(axis=-1)
    return out

In [12]:
import numpy as np
all_songs = df.drop_duplicates('song_id')
all_songs = all_songs.set_index('song_id')[list(user_emb.columns)]
# матрицы в одинаковом порядке колонок!
M = all_songs.values.astype(np.float64)
P = user_emb.values.astype(np.float64)

# # нормы для cosine
# M_norm = np.linalg.norm(M, axis=1) + 1e-9
# P_norm = np.linalg.norm(P, axis=1) + 1e-9

# # cosine similarity — все юзеры против всех песен
# scores = P @ M.T                                # (n_users, n_songs)
scores = cosine_similarity(P, M)

# --- маскируем прослушанные ---
song_ids = all_songs.index.values
song_to_col = {s: i for i, s in enumerate(song_ids)}

user_ids_order = user_emb.index.values
user_to_row = {u: i for i, u in enumerate(user_ids_order)}

# train — исходный DataFrame взаимодействий из train split
for u, grp in train.groupby('user_id'):
    row = user_to_row.get(u)
    if row is None:
        continue
    cols = [song_to_col[s] for s in grp['song_id'] if s in song_to_col]
    scores[row, cols] = -np.inf


In [13]:
K = 10
top_idx = np.argpartition(-scores, K, axis=1)[:, :K]
# досортировать только эти K столбцов
row_idx = np.arange(len(scores))[:, None]
order = np.argsort(-scores[row_idx, top_idx], axis=1)
top_idx = top_idx[row_idx, order]
song_ids = all_songs.index.to_numpy()
top_song_ids = song_ids[top_idx]   # (n_users, K) — финальные рекомендации
top_song_ids

array([['SOHUGUY12A6D4F859C', 'SOUDQDW12AF729F367', 'SOOTCFE12A6310F231',
        ..., 'SOYORHQ12A8C1436A4', 'SOUTBAG12A8C138320',
        'SOJRDNG12A6D4F9119'],
       ['SOTRQEJ12AF72A45D7', 'SOVJXJU12A6310E226', 'SOZONZI12A58A7D32F',
        ..., 'SOSRIAT12AB017A92E', 'SORRYFP12A8C13A08D',
        'SONIIGT12A58A78884'],
       ['SOFCKUP12A6310E31D', 'SOFMZDX12A670208FB', 'SOAYATB12A6701FD50',
        ..., 'SOQAUUD12A8C1344E3', 'SOOKORP12AF72A6DE2',
        'SOVGSEL12A8C141591'],
       ...,
       ['SOWOZBR12A8C13F4F6', 'SOQQTFV12A6701C5F6', 'SOAKCFL12A6D4F9CC5',
        ..., 'SOGSBWS12AB01858D0', 'SOJPFPR12AB018109D',
        'SOIYVLG12AAFF43E95'],
       ['SOUZRZK12A8C13FF59', 'SOITSDN12A8C1359F8', 'SOFGDGA12AB017C86B',
        ..., 'SOBBONW12AB0180FE6', 'SOUHNQN12AF72A3DE3',
        'SORKPAQ12AB018391F'],
       ['SOXOEUD12AB018CF4C', 'SOYLKVN12A6D4F7CCA', 'SOKVTGU12A6701E7B1',
        ..., 'SOUOECO12A6D4F5C6F', 'SOYTKDP12A6D223C06',
        'SOANSPC12A6702154A']], shape=(24064, 1

In [16]:
test_truth = test.groupby('user_id')['song_id'].apply(set).to_dict()

## сравнение метрик расстояния

In [17]:
# индексы train взаимодействий — один раз, чтобы маскировать каждую матрицу O(1)
train_rows, train_cols = [], []
for u, grp in train.groupby('user_id'):
    row = user_to_row.get(u)
    if row is None:
        continue
    for s_id in grp['song_id']:
        col = song_to_col.get(s_id)
        if col is not None:
            train_rows.append(row)
            train_cols.append(col)
train_rows = np.array(train_rows)
train_cols = np.array(train_cols)

K = 10
song_ids_arr = all_songs.index.to_numpy()

def evaluate_method(scores_matrix, higher_is_better):
    fill_value = -np.inf if higher_is_better else np.inf
    s = scores_matrix.copy()
    s[train_rows, train_cols] = fill_value

    if higher_is_better:
        idx = np.argpartition(-s, K, axis=1)[:, :K]
        order = np.argsort(-s[np.arange(len(s))[:, None], idx], axis=1)
    else:
        idx = np.argpartition(s, K, axis=1)[:, :K]
        order = np.argsort(s[np.arange(len(s))[:, None], idx], axis=1)
    idx = idx[np.arange(len(s))[:, None], order]
    top = song_ids_arr[idx]

    hits, precs, recs = [], [], []
    for i, u in enumerate(user_ids_order):
        truth = test_truth.get(u, set())
        if not truth:
            continue
        h = sum(1 for s_id in top[i] if s_id in truth)
        hits.append(h)
        precs.append(h / K)
        recs.append(h / len(truth))
    return np.mean(hits), np.mean(precs), np.mean(recs)

In [18]:
from sklearn.preprocessing import normalize
P_n = normalize(P, axis=1)
M_n = normalize(M, axis=1)

methods = [
    ('cosine_raw',    lambda: cosine_similarity(P, M),       True),
    ('cosine_norm',   lambda: cosine_similarity(P_n, M_n),   True),  # ≡ dot на P_n,M_n
    ('euclidean_raw',  lambda: euclideam_dist(P, M),         False),
    ('euclidean_norm', lambda: euclideam_dist(P_n, M_n),     False),
    ('manhattan_raw',  lambda: manhattan_dist(P, M),         False),
    ('manhattan_norm', lambda: manhattan_dist(P_n, M_n),     False),
    ('chebyshev_raw',  lambda: chebyshev(P, M),              False),
    ('chebyshev_norm', lambda: chebyshev(P_n, M_n),          False),
]


rows = []
for name, fn, higher in methods:
    print(f'-> {name}')
    scores_m = fn()
    h, p, r = evaluate_method(scores_m, higher_is_better=higher)
    rows.append({'method': name, 'mean_hits': h, f'precision@{K}': p, f'recall@{K}': r})
    del scores_m

pd.DataFrame(rows)

-> cosine_raw
-> cosine_norm
-> euclidean_raw
-> euclidean_norm
-> manhattan_raw
-> manhattan_norm
-> chebyshev_raw
-> chebyshev_norm


,method,mean_hits,precision@10,recall@10
0,cosine_raw,0.031037,0.003104,0.007974
1,cosine_norm,0.031037,0.003104,0.007974
2,euclidean_raw,0.014959,0.001496,0.003921
3,euclidean_norm,0.031037,0.003104,0.007974
4,manhattan_raw,0.015093,0.001509,0.004270
5,manhattan_norm,0.030948,0.003095,0.008184
6,chebyshev_raw,0.011286,0.001129,0.002642
7,chebyshev_norm,0.023200,0.002320,0.005243


In [19]:
# --- Сравнение метрик через utils.distance_metrics ---
import sys, os
sys.path.insert(0, os.path.expanduser('~/kursach'))
from utils.distance_metrics import evaluate_all, to_dataframe
from sklearn.preprocessing import normalize

# id в порядке строк P (юзеры) и M (песни)
user_ids_order = user_emb.index.to_numpy()
song_ids_order = all_songs.index.to_numpy()

# словари из train/test split
train_seen   = train.groupby('user_id')['song_id'].apply(set).to_dict()
test_by_user = test.groupby('user_id')['song_id'].apply(set).to_dict()

# --- RAW: содержимое признаков несёт сигнал, dot/L2/L1 будут разные ---
results_audio_raw = evaluate_all(
    P=P, M=M,
    user_ids=user_ids_order, item_ids=song_ids_order,
    train_seen=train_seen, test_truth=test_by_user, K=10,
)

# --- NORMALIZED: единичные векторы, dot ≡ cos ≡ L2 по ranking-у ---
P_n = normalize(P, axis=1)
M_n = normalize(M, axis=1)
results_audio_norm = evaluate_all(
    P=P_n, M=M_n,
    user_ids=user_ids_order, item_ids=song_ids_order,
    train_seen=train_seen, test_truth=test_by_user, K=10,
)


[13:40:26] [1/7] metric 'dot': computing scores...
[13:40:27]   scores computed in 0.8s, shape=(24064, 10540)
[13:40:27]   copy scores (24064x10540) -> float64
[13:40:35]   masking train interactions...
[13:40:41]   masking done in 5.8s, computing top-K + metrics...
[13:40:42]     5000/24064 users  rate=5714/s  eta=3s
[13:40:43]     10000/24064 users  rate=6859/s  eta=2s
[13:40:43]     15000/24064 users  rate=7491/s  eta=1s
[13:40:44]     20000/24064 users  rate=8033/s  eta=1s
[13:40:44]   done: 22328 users evaluated in 2.9s
[13:40:44]   'dot' result: P@10=0.0016 R@10=0.0038 NDCG@10=0.0027
[13:40:44] [2/7] metric 'cos': computing scores...
[13:40:45]   scores computed in 0.7s, shape=(24064, 10540)
[13:40:45]   copy scores (24064x10540) -> float64
[13:41:03]   masking train interactions...
[13:41:16]   masking done in 13.3s, computing top-K + metrics...
[13:41:17]     5000/24064 users  rate=6871/s  eta=3s
[13:41:17]     10000/24064 users  rate=7096/s  eta=2s
[13:41:18]     15000/24064 u

In [20]:
# --- сводная таблица ---
table = pd.concat({
    'raw':  to_dataframe(results_audio_raw),
    'norm': to_dataframe(results_audio_norm),
}, names=['mode', 'metric'])
print(table)
table.to_csv('results_audio.csv')


             precision@10  recall@10   ndcg@10  users_evaluated
mode metric                                                    
raw  dot         0.001568   0.003754  0.002748          22328.0
     cos         0.003104   0.007974  0.006079          22328.0
     l2          0.001496   0.003921  0.002877          22328.0
     l1          0.001509   0.004270  0.002948          22328.0
     linf        0.001111   0.002601  0.002039          22328.0
     lp_0.5      0.001491   0.004331  0.002886          22328.0
     maha        0.002074   0.006302  0.004344          22328.0
norm dot         0.003104   0.007974  0.006079          22328.0
     cos         0.003104   0.007974  0.006079          22328.0
     l2          0.003104   0.007974  0.006079          22328.0
     l1          0.003095   0.008184  0.006064          22328.0
     linf        0.002320   0.005243  0.004236          22328.0
     lp_0.5      0.002817   0.007292  0.005558          22328.0
     maha        0.002651   0.007633  0.